# Text2LoRA / Doc2LoRA — generate LoRA weights from a task description or document, no training (`core/adapters/text2lora`)

In [ ]:
import torch
from aligntune.core.adapters import TextToLoRAHypernet, DocToLoRA

# TextToLoRA: a hypernetwork that maps a task-description embedding straight
# to LoRA A/B matrices -- no per-task LoRA training loop required.
hypernet = TextToLoRAHypernet(
    hidden_dim=384,           # matches all-MiniLM-L6-v2 sentence-transformer output
    lora_r=16,
    num_target_modules=4,     # e.g. q_proj, k_proj, v_proj, o_proj
)

# get_embedding_model() returns a SentenceTransformer for embedding task text;
# the hypernet itself only consumes the resulting [batch, hidden_dim] tensor.
embedding_model = hypernet.get_embedding_model()
task_description = "Answer banking customer-support questions about KYC and account opening."
task_embedding = torch.tensor(embedding_model.encode([task_description]))

lora_weights = hypernet(task_embedding)
print("Generated LoRA pairs:", len(lora_weights))
print("A shape:", lora_weights[0]["A"].shape, "B shape:", lora_weights[0]["B"].shape)

In [ ]:
# Doc2LoRA: same hypernetwork, but for a long document instead of a short
# description -- chunks the text, embeds + pools the chunks, then generates
# LoRA weights from the pooled embedding. Useful for per-document adapters
# (e.g. one LoRA per SEC filing or per compliance policy) with zero training.
doc2lora = DocToLoRA(hypernet, chunk_size=512, num_chunks=3, pooling_strategy="mean")

long_doc = (
    "This retail banking compliance policy document describes KYC "
    "verification steps, mandatory disclosures, and escalation procedures "
    "for suspected fraud... " * 20
)
doc_lora_weights = doc2lora(long_doc)
print("Generated LoRA pairs from document:", len(doc_lora_weights))